# Public Names for one Test and their count

In [ ]:
test_date = "2024-11-07"

In [ ]:
import os
import pickle
from datetime import datetime

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy.pool import QueuePool

# Load environment variables from .env file
load_dotenv("../../.env")  # Adjust the path if your .env file is in a different location


def fetch_and_save_data(query, filename_suffix, test_date, params=None, timeout=3600):
    """
    Fetches data from PostgreSQL using SQLAlchemy with connection pooling,
    saves it to a pickle file, and handles timeouts.
    Reads database credentials from environment variables.

    Args:
        query (str): The SQL query to execute.
        filename_suffix (str): A string to be added to the pickle filename.
        params (tuple, optional): Parameters to pass to the query. Defaults to None.
        timeout (int, optional): The timeout for the database connection in seconds. Defaults to 3600.
    """
    try:
        db_user = os.getenv("DB_USER")
        db_password = os.getenv("DB_PASSWORD")
        db_host = os.getenv("DB_HOST")
        db_name = os.getenv("DB_NAME")

        engine = create_engine(
            f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}/{db_name}",
            connect_args={"connect_timeout": timeout},
            poolclass=QueuePool,
            pool_size=5,
            max_overflow=10,
        )

        with engine.connect() as conn:
            df = pd.read_sql_query(query, conn, params=(test_date,))
        print(df)
        os.makedirs("./pickle", exist_ok=True)
        filename = f"./pickle/{datetime.now().strftime('%Y%m%d_%H%M%S')}_{filename_suffix}.pickle"

        with open(filename, "wb") as f:
            pickle.dump(df, f)

        print(f"Data saved to {filename}")

    except Exception as e:
        print(f"Error: {e}")


if __name__ == "__main__":
    query = """
SELECT
    hr.test_date,
    ec.ech_public_name,
    COUNT(hr.ech_config) AS total_ech_configs
FROM
    public."HTTPSRecords" hr
JOIN
    public."HTTPSRecordsECHConfigs" hrec ON hr.id = hrec.https_record_id
JOIN
    public."ECHConfigs" ec ON hrec.ech_config_id = ec.id
WHERE
    hr.test_date = %s
    AND hr.ech_config IS NOT NULL
    AND hr.ech_config != ''
GROUP BY
    hr.test_date, ec.ech_public_name
ORDER BY
    hr.test_date, ec.ech_public_name;
    """
    filename_suffix = "ech_ciphe-suites_by_test_relation"
    fetch_and_save_data(query, filename_suffix, test_date)

# Publicnames by ARecords

In [ ]:
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv("../../.env")  # Adjust the path if your .env file is in a different location


def fetch_and_save_data(query, filename_suffix, test_date, params=None, timeout=3600):
    """
    Fetches data from PostgreSQL using SQLAlchemy with connection pooling,
    saves it to a pickle file, and handles timeouts.
    Reads database credentials from environment variables.

    Args:
        query (str): The SQL query to execute.
        filename_suffix (str): A string to be added to the pickle filename.
        params (tuple, optional): Parameters to pass to the query. Defaults to None.
        timeout (int, optional): The timeout for the database connection in seconds. Defaults to 3600.
    """
    try:
        db_user = os.getenv("DB_USER")
        db_password = os.getenv("DB_PASSWORD")
        db_host = os.getenv("DB_HOST")
        db_name = os.getenv("DB_NAME")

        engine = create_engine(
            f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}/{db_name}",
            connect_args={"connect_timeout": timeout},
            poolclass=QueuePool,
            pool_size=5,
            max_overflow=10,
        )

        with engine.connect() as conn:
            df = pd.read_sql_query(query, conn, params=(test_date,))
        print(df)
        os.makedirs("./pickle", exist_ok=True)
        filename = f"./pickle/{datetime.now().strftime('%Y%m%d_%H%M%S')}_{filename_suffix}.pickle"

        with open(filename, "wb") as f:
            pickle.dump(df, f)

        print(f"Data saved to {filename}")

    except Exception as e:
        print(f"Error: {e}")


if __name__ == "__main__":
    query = """
SELECT
    dr.test_code,
    ar.asn,
    ar.asn_org,
    ar.asn_country,
    ar.asn_city,
    ec.ech_public_name,
    COUNT(DISTINCT hr.domain) AS ech_config_count
FROM
    public."DNSResults" dr
JOIN
    public."DNSResultsARecords" dra ON dr.id = dra.dns_result_id  -- Changed to DNSResultsARecords
JOIN
    public."ARecords" ar ON dra.a_record_id = ar.id  -- Changed to ARecords
JOIN
    public."DNSResultsHTTPSRecords" drhr ON dr.id = drhr.dns_result_id
JOIN
    public."HTTPSRecords" hr ON drhr.https_record_id = hr.id
JOIN
    public."HTTPSRecordsECHConfigs" hrec ON hr.id = hrec.https_record_id
JOIN
    public."ECHConfigs" ec ON hrec.ech_config_id = ec.id
WHERE
    hr.test_date = %s
    AND hr.ech_config IS NOT NULL
    AND hr.ech_config != ''
GROUP BY
    dr.test_code, ar.asn, ar.asn_org, ar.asn_country, ar.asn_city, ec.ech_public_name
ORDER BY
    dr.test_code, ar.asn, ar.asn_org, ar.asn_country, ar.asn_city, ec.ech_public_name;
    """
    filename_suffix = "ech_a_relation_single_test"
    fetch_and_save_data(query, filename_suffix, test_date)

# Publicnames by AAAARecords

In [ ]:
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv("../.env")  # Adjust the path if your .env file is in a different location


def fetch_and_save_data(query, filename_suffix, params=None, timeout=3600):
    """
    Fetches data from PostgreSQL using SQLAlchemy with connection pooling,
    saves it to a pickle file, and handles timeouts.
    Reads database credentials from environment variables.

    Args:
        query (str): The SQL query to execute.
        filename_suffix (str): A string to be added to the pickle filename.
        params (tuple, optional): Parameters to pass to the query. Defaults to None.
        timeout (int, optional): The timeout for the database connection in seconds. Defaults to 3600.
    """
    try:
        db_user = os.getenv("DB_USER")
        db_password = os.getenv("DB_PASSWORD")
        db_host = os.getenv("DB_HOST")
        db_name = os.getenv("DB_NAME")

        engine = create_engine(
            f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}/{db_name}",
            connect_args={"connect_timeout": timeout},
            poolclass=QueuePool,
            pool_size=5,
            max_overflow=10,
        )

        with engine.connect() as conn:
            df = pd.read_sql_query(query, conn, params=(test_date,))

        os.makedirs("./pickle", exist_ok=True)
        filename = f"./pickle/{datetime.now().strftime('%Y%m%d_%H%M%S')}_{filename_suffix}.pickle"

        with open(filename, "wb") as f:
            pickle.dump(df, f)

        print(f"Data saved to {filename}")

    except Exception as e:
        print(f"Error: {e}")


if __name__ == "__main__":
    query = """
SELECT
    dr.test_code,
    ar.asn,
    ar.asn_org,
    ar.asn_country,
    ar.asn_city,
    ec.ech_public_name,
    COUNT(DISTINCT hr.domain) AS ech_config_count
FROM
    public."DNSResults" dr
JOIN
    public."DNSResultsAAAARecords" dra ON dr.id = dra.dns_result_id  -- Changed to DNSResultsAAAARecords
JOIN
    public."AAAARecords" ar ON dra.aaaa_record_id = ar.id  -- Changed to AAAARecords
JOIN
    public."DNSResultsHTTPSRecords" drhr ON dr.id = drhr.dns_result_id
JOIN
    public."HTTPSRecords" hr ON drhr.https_record_id = hr.id
JOIN
    public."HTTPSRecordsECHConfigs" hrec ON hr.id = hrec.https_record_id
JOIN
    public."ECHConfigs" ec ON hrec.ech_config_id = ec.id
WHERE
    hr.test_date = %s
    AND hr.ech_config IS NOT NULL
    AND hr.ech_config != ''
GROUP BY
    dr.test_code, ar.asn, ar.asn_org, ar.asn_country, ar.asn_city, ec.ech_public_name
ORDER BY
    dr.test_code, ar.asn, ar.asn_org, ar.asn_country, ar.asn_city, ec.ech_public_name;
    """
    filename_suffix = "ech_aaaa_relation_single_test"
    fetch_and_save_data(query, filename_suffix, test_date)

# Publicnames by Authoritative Nameservers

In [ ]:
from dotenv import load_dotenv

test_date = "2024-11-07"
# Load environment variables from .env file
load_dotenv("../.env")  # Adjust the path if your .env file is in a different location


def fetch_and_save_data(query, filename_suffix, test_date, params=None, timeout=3600):
    """
    Fetches data from PostgreSQL using SQLAlchemy with connection pooling,
    saves it to a pickle file, and handles timeouts.
    Reads database credentials from environment variables.

    Args:
        query (str): The SQL query to execute.
        filename_suffix (str): A string to be added to the pickle filename.
        params (tuple, optional): Parameters to pass to the query. Defaults to None.
        timeout (int, optional): The timeout for the database connection in seconds. Defaults to 3600.
    """
    try:
        db_user = os.getenv("DB_USER")
        db_password = os.getenv("DB_PASSWORD")
        db_host = os.getenv("DB_HOST")
        db_name = os.getenv("DB_NAME")

        engine = create_engine(
            f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}/{db_name}",
            connect_args={"connect_timeout": timeout},
            poolclass=QueuePool,
            pool_size=5,
            max_overflow=10,
        )

        with engine.connect() as conn:
            df = pd.read_sql_query(query, conn, params=(test_date,))

        os.makedirs("./pickle", exist_ok=True)
        filename = f"./pickle/{datetime.now().strftime('%Y%m%d_%H%M%S')}_{filename_suffix}.pickle"

        with open(filename, "wb") as f:
            pickle.dump(df, f)

        print(f"Data saved to {filename}")

    except Exception as e:
        print(f"Error: {e}")


if __name__ == "__main__":
    query = """
SELECT
    dr.test_code,
    ar.asn,
    ar.asn_org,
    ar.asn_country,
    ar.asn_city,
    ec.ech_public_name,
    COUNT(DISTINCT hr.domain) AS ech_config_count  -- Changed to count distinct domains
FROM
    public."DNSResults" dr
JOIN
    public."DNSResultsAuthoritativeNameserverRecords" dra ON dr.id = dra.dns_result_id
JOIN
    public."AuthoritativeNameserverRecords" ar ON dra.authoritative_nameserver_record_id = ar.id
JOIN
    public."DNSResultsHTTPSRecords" drhr ON dr.id = drhr.dns_result_id
JOIN
    public."HTTPSRecords" hr ON drhr.https_record_id = hr.id
JOIN
    public."HTTPSRecordsECHConfigs" hrec ON hr.id = hrec.https_record_id
JOIN
    public."ECHConfigs" ec ON hrec.ech_config_id = ec.id
WHERE
    hr.test_date = %s
    AND hr.ech_config IS NOT NULL  -- Only filter on hr.ech_config
    AND hr.ech_config != ''
GROUP BY
    dr.test_code, ar.asn, ar.asn_org, ar.asn_country, ar.asn_city, ec.ech_public_name
ORDER BY
    dr.test_code, ar.asn, ar.asn_org, ar.asn_country, ar.asn_city, ec.ech_public_name;
    """
    filename_suffix = "ech_authoritative-ns_relation_single_test"
    fetch_and_save_data(query, filename_suffix, test_date)